# **IMPORT LIBRARY**

Tahap ini dilakukan untuk mengimpor berbagai library yang akan diperlukan untuk mendukung proses analisis data.

In [9]:
import os
import pandas as pd
import re

from num2words import num2words

# **LOAD DATA**

Tahap ini dilakukan untuk memuat *(load)* dataset yang akan digunakan dalam proses analisis.

In [2]:
df = pd.read_csv("../data/processed/transactions/transactions_clean.csv")

In [3]:
# Cek isi data dalam 5 baris awal
df.head()

,Transaction_ID,Date,Product_Name,Category,Units_Sold,Unit_Price,Revenue,Stock_In,Stock_Out,Stock_End
0,TRX000001,2023-01-01,ABC Kecap Asin 620ml,Groceries,47,18000.0,846000.0,627,47,580
1,TRX000002,2023-01-01,ABC Kecap Manis 620ml,Groceries,13,18000.0,234000.0,486,13,473
2,TRX000003,2023-01-01,ABC Sari Kacang Hijau 250ml,Drinks,2,4000.0,8000.0,673,2,671
3,TRX000004,2023-01-01,Aqua 600ml,Drinks,7,4000.0,28000.0,690,7,683
4,TRX000005,2023-01-01,Bango Kecap Manis 189gr,Groceries,19,18000.0,342000.0,674,19,655


**Insight:**

- Dataset berhasil dimuat ke lingkungan kerja dan terbaca dalam bentuk *dataframe*, sehingga siap digunakan pada proses analisis selanjutnya.
- Hasil pemeriksaan awal menggunakan `df.head()` menunjukkan bahwa data berhasil ditampilkan dan setiap kolom dapat terbaca dengan baik, sehingga dapat digunakan untuk proses analisis selanjutnya.


# **DATA PREPARATION**

Tahap ini dilakukan untuk mempersiapkan data produk yang akan digunakan pada proses transkripsi. Proses ini meliputi pengecekan tipe data, pengambilan data produk unik beserta harga satuan pengurutan data berdasarkan nama produk, serta pengaturan ulang indeks agar data lebih terstruktur dan memudahkan penggunaan pada tahap selanjutnya.

In [4]:
# Cek type data
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 63161 entries, 0 to 63160
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Transaction_ID  63161 non-null  str    
 1   Date            63161 non-null  str    
 2   Product_Name    63161 non-null  str    
 3   Category        63161 non-null  str    
 4   Units_Sold      63161 non-null  int64  
 5   Unit_Price      63161 non-null  float64
 6   Revenue         63161 non-null  float64
 7   Stock_In        63161 non-null  int64  
 8   Stock_Out       63161 non-null  int64  
 9   Stock_End       63161 non-null  int64  
dtypes: float64(2), int64(4), str(4)
memory usage: 7.5 MB


In [5]:
# Ambil hanya product unik
df = df[['Product_Name', 'Unit_Price']].drop_duplicates(subset='Product_Name')

# Urutkan berdasarkan abjad dan reset index
df = df.sort_values(by='Product_Name').reset_index(drop=True)

# Cek dataset
df

,Product_Name,Unit_Price
0,ABC Kecap Asin 620ml,18000.0
1,ABC Kecap Manis 620ml,18000.0
2,ABC Sari Kacang Hijau 250ml,4000.0
3,Acnes Obat Jerawat,20000.0
4,Antimo Obat Anti Mabuk,7000.0
...,...,...
115,Tropicana Slim,45000.0
116,Wafello Wafer,5000.0
117,Wardah Cushion,90000.0
118,Wipol 750ml,25000.0


**Inisight:**

Data produk berhasil dipersiapkan dengan mempertahankan produk unik beserta harga satuannya sehingga dapat digunakan pada proses transkripsi. Pengurutan data berdasarkan nama produk dan pengaturan ulang indeks menghasilkan struktur data yang lebih rapi serta memudahkan pencarian informasi produk.

# **TRANSCRIPTS**

Tahap ini dilakkukan untuk menyusun data transkrip berbasis teks yang digunakan sebagai pendukung proses audio dan pengembangan sistem pencatatan transaksi berbasis *voice input*. Proses ini meliputi penyesuaian nama produk, konversi satuan dan angka ke bentuk teks, pembentukan variasi transaksi berdasarkan jumlah pembelian, penggunaan variasi penyebutan transaksi, serta penyusunan kalimat hasil transkrip ke dalam format yang lebih terstruktur.

In [6]:
from typing import Text
# Mengubah angka menjadi teks
def number_to_text(number):
  return num2words(int(number), lang='id')

# Normalize unit
def normalize_unit(text):
  # Kasih spasi antara angka dan huruf
  text = re.sub(r'(\d+)([a-zA-Z]+)', r'\1 \2', text)
  units = {
      'ml': 'mililiter',
      'gr': 'gram',
      'kg': 'kilogram',
      'l': 'liter'
  }
  for short, full in units.items():
    text = re.sub(
        rf'\b{short}\b',
        full,
        text,
        flags=re.IGNORECASE
    )
  return text

# Mengubah angka ke dalam teks
def convert_number_in_text(text):
  return re.sub(
      r'\d+',
      lambda match: number_to_text(match.group()),
      text
  )

In [7]:
# Generate transcript
new_data = []
nomor_file = 1

for _, row in df.iterrows():
  # Lowercase Product_Name
  product = row['Product_Name'].lower()

  # Ubah glad2glow menjadi glad two glow dan C1000 menjadi c thousand
  product = re.sub(r'glad2glow', 'glad two glow', product, flags=re.IGNORECASE)
  product = re.sub(r'c1000', 'c thousand', product, flags=re.IGNORECASE)

  # Rapihin spasi
  product = re.sub(r'\s+', ' ', product).strip()

  # Normalize unit
  product = normalize_unit(product)

  # Ubah angka di Product_Name
  product = convert_number_in_text(product)
  price = int(row['Unit_Price'])

  for quantity in range(1, 11):
    # Copy product
    final_product = product

    # Hapus nama product hanya transcript tertentu
    if nomor_file in [1, 9, 10]:
      final_product = re.sub(r'\babc\b', '', final_product)

    final_product = re.sub(r'\s+', ' ', final_product).strip()
    total_price = price * quantity

    # Default keyword
    keyword = "jual"

    # List keyword laku
    if nomor_file in (
        list(range(161, 181)) +
        list(range(211, 221)) +
        list(range(241, 251)) +
        list(range(551, 561)) +
        list(range(571, 591)) +
        list(range(631, 641)) +
        list(range(831, 841)) +
        list(range(861, 871)) +
        list(range(891, 901))
      ):
      keyword = "laku"

    # List keyword keluar
    elif nomor_file in (
        list(range(141, 151)) +
        list(range(181, 191)) +
        list(range(221, 231)) +
        list(range(561, 571)) +
        list(range(591, 601)) +
        list(range(611, 621)) +
        list(range(841, 851)) +
        list(range(871, 881))
      ):
      keyword = "keluar"

    transcript = (
        f"{keyword} {final_product} "
        f"{number_to_text(quantity)} bungkus "
        f"{number_to_text(total_price)}"
    )
    new_data.append({
        "filename": f"{nomor_file}.wav",
        "transcript": transcript
    })
    nomor_file += 1

In [8]:
# Simpan dalam bentuk DataFrame
result = pd.DataFrame(new_data)

# Cek isi kolom
result.head(20)

,filename,transcript
0,1.wav,jual kecap asin enam ratus dua puluh mililiter...
1,2.wav,jual abc kecap asin enam ratus dua puluh milil...
2,3.wav,jual abc kecap asin enam ratus dua puluh milil...
3,4.wav,jual abc kecap asin enam ratus dua puluh milil...
4,5.wav,jual abc kecap asin enam ratus dua puluh milil...
5,6.wav,jual abc kecap asin enam ratus dua puluh milil...
6,7.wav,jual abc kecap asin enam ratus dua puluh milil...
7,8.wav,jual abc kecap asin enam ratus dua puluh milil...
8,9.wav,jual kecap asin enam ratus dua puluh mililiter...
9,10.wav,jual kecap asin enam ratus dua puluh mililiter...


In [10]:
# Save ke CSV
os.makedirs("../data/processed/nlp", exist_ok=True)
result.to_csv("../data/processed/nlp/transcripts.csv", index=False)

**Insight:**

Data transkrip berhasil disusun dengan membentuk variasi kalimat berdasarkan nama produk, jumlah pembelian, dan total harga. Proses penyesuaian nama produk, perubahan angka menjadi teks, serta penggunaan variasi penyebutan transaksi seperti jual, laku, dan keluar membantu menghasilkan format transkrip yang lebih beragam untuk kebutuhan pemrosesan audio.

# **NLP LABELING**

Tahap ini dilakukan untuk membangun dataset *Natural Language Processing* (NLP) dari data transkrip dengan menambahkan label *intent* dan *entity*. Proses ini meliputi identifikasi jenis transaksi berdasarkan variasi penyebutan transaksi yang digunakan, ekstraksi informasi penting, serta penyusunan data berlabel NLU dalam format terstruktur untuk kebutuhan pemrosesan audio.

In [11]:
# Membuat intent
def create_intent(text):
  text = str(text).lower().strip()
  if text.startswith("jual"):
    return "jual"
  elif text.startswith("laku"):
    return "laku"
  elif text.startswith("keluar"):
    return "keluar"
  else:
    return "unknown"

In [12]:
# Membuat entity
def create_entities(text):
  text = str(text).lower()
  entities = []

  # Jumlah produk
  jumlah_pattern = (
      r"(satu|dua|tiga|empat|lima|enam|"
      r"tujuh|delapan|sembilan|sepuluh)\sbungkus"
  )
  jumlah_match = re.search(
      jumlah_pattern,
      text
  )
  if jumlah_match:
    entities.append({
        "entity": "jumlah",
        "value": jumlah_match.group()
    })

  # Harga produk
  harga_pattern = r"bungkus\s(.+)$"
  harga_match = re.search(
      harga_pattern,
      text
  )
  if harga_match:
    entities.append({
        "entity": "harga",
        "value": harga_match.group(1)
    })

  # Produk
  produk_pattern = (
      r"(?:jual|laku|keluar)\s"
      r"(.+?)\s"
      r"(?:satu|dua|tiga|empat|lima|enam|"
      r"tujuh|delapan|sembilan|sepuluh)\sbungkus"
  )
  produk_match = re.search(
    produk_pattern,
    text
  )
  if produk_match:
    entities.append({
        "entity": "produk",
        "value": produk_match.group(1)
    })
  return entities

In [13]:
# Membuat label NLU
nlu_df = result.copy()
nlu_df["intent"] = nlu_df["transcript"].apply(
    create_intent
)
nlu_df["entities"] = nlu_df["transcript"].apply(
    create_entities
)

In [63]:
# Cek isi dataset
nlu_df.head(10)

,filename,transcript,intent,entities
0,1.wav,jual kecap asin enam ratus dua puluh mililiter...,jual,"[{'entity': 'jumlah', 'value': 'satu bungkus'}..."
1,2.wav,jual abc kecap asin enam ratus dua puluh milil...,jual,"[{'entity': 'jumlah', 'value': 'dua bungkus'},..."
2,3.wav,jual abc kecap asin enam ratus dua puluh milil...,jual,"[{'entity': 'jumlah', 'value': 'tiga bungkus'}..."
3,4.wav,jual abc kecap asin enam ratus dua puluh milil...,jual,"[{'entity': 'jumlah', 'value': 'empat bungkus'..."
4,5.wav,jual abc kecap asin enam ratus dua puluh milil...,jual,"[{'entity': 'jumlah', 'value': 'lima bungkus'}..."
5,6.wav,jual abc kecap asin enam ratus dua puluh milil...,jual,"[{'entity': 'jumlah', 'value': 'enam bungkus'}..."
6,7.wav,jual abc kecap asin enam ratus dua puluh milil...,jual,"[{'entity': 'jumlah', 'value': 'tujuh bungkus'..."
7,8.wav,jual abc kecap asin enam ratus dua puluh milil...,jual,"[{'entity': 'jumlah', 'value': 'delapan bungku..."
8,9.wav,jual kecap asin enam ratus dua puluh mililiter...,jual,"[{'entity': 'jumlah', 'value': 'sembilan bungk..."
9,10.wav,jual kecap asin enam ratus dua puluh mililiter...,jual,"[{'entity': 'jumlah', 'value': 'sepuluh bungku..."


In [14]:
# Save ke CSV
nlu_df.to_csv("../data/processed/nlp/nlu_labeled.csv", index=False)

**Insight:**

Dataset NLP berhasil dibentuk dengan menambahkan label *intent* dan *entity* pada data transkrip. Proses ini memungkinkan identifikasi jenis transaksi serta mengekstraksi informasi penting seperti nama produk, jumlah pembelian, dan harga sehingga dataset menjadi lebih terstruktur dan siap digunakan untuk proses pemodelan dan pengembangan sistem berbasis *voice input*.